# Error Analysis — Per-class, Subgroup, and Confusion Patterns

This notebook assumes a trained BERT model is available (via `scripts/run_finetune.py --model bert`)
and that `results/d3_logs.jsonl` exists (via `scripts/run_d3_agentic.py --save-logs`).

The analysis cells are kept close to the originals used in the project reports
(per-class error rates, answer-length quartiles, confusion matrix inspection).

## 1. Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Per-class F1 table (reads results/metrics.csv)

In [ ]:
metrics = pd.read_csv("../results/metrics.csv")
metrics[[c for c in metrics.columns if c.startswith("f1_") or c in ("method", "split")]]

## 3. Confusion matrices from BERT (ported from the fine-tune notebook)

In [ ]:
cm = confusion_matrix(val_labels, val_preds)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names
)
plt.title(f'Confusion Matrix — {MODEL_NAME} (Validation)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_bert.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Error rate by answer-length quartile (BERT, from TN2.2 notebook)

In [ ]:
# ── Error rate by answer length quartile ─────────────────────────────────────
df_errors['a_len_quartile'] = pd.qcut(df_errors['a_len'], q=4, labels=['Q1 (short)', 'Q2', 'Q3', 'Q4 (long)'])
error_rate_by_len = df_errors.groupby('a_len_quartile', observed=True)['correct'].apply(
    lambda x: 1 - x.mean()
).reset_index()
error_rate_by_len.columns = ['Answer Length Quartile', 'Error Rate']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(
    error_rate_by_len['Answer Length Quartile'],
    error_rate_by_len['Error Rate'],
    color='salmon'
)
axes[0].set_title('Error Rate by Answer Length Quartile')
axes[0].set_ylabel('Error Rate')
axes[0].grid(axis='y', alpha=0.3)

# Per-class error counts
axes[1].bar(
    errors['true'].value_counts().index,
    errors['true'].value_counts().values,
    color='steelblue'
)
axes[1].set_title('Error Count by True Label')
axes[1].set_ylabel('Number of Errors')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('error_analysis_bert.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Concrete misclassifications (BERT, from TN2.2 notebook)

In [ ]:
# ── Concrete error examples ───────────────────────────────────────────────────
print('Sample Errors — Clear Reply predicted as Ambivalent:')
print('=' * 80)
sample = errors[(errors['true'] == 'Clear Reply') & (errors['pred'] == 'Ambivalent')].head(3)
for _, row in sample.iterrows():
    print(f'Q: {row["question"][:150]}')
    print(f'A: {row["interview_answer"][:300]}')
    print(f'True: {row["true"]}  |  Pred: {row["pred"]}')
    print('-' * 80)

print('\nSample Errors — Clear Non-Reply predicted as Ambivalent:')
print('=' * 80)
sample2 = errors[(errors['true'] == 'Clear Non-Reply') & (errors['pred'] == 'Ambivalent')].head(3)
for _, row in sample2.iterrows():
    print(f'Q: {row["question"][:150]}')
    print(f'A: {row["interview_answer"][:300]}')
    print(f'True: {row["true"]}  |  Pred: {row["pred"]}')
    print('-' * 80)